# RO change over time

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar

# import gsw

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Functions

In [ ]:
def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_stats(x):
    """helper function to compute plotting bounds for experiment"""
    stats = x.quantile(q=[0.1, 0.5, 0.9], dim="member")
    return stats.rename({"quantile": "q"})


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## full path to file
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## load if it already exists
        if fp.is_file():
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def save(fig, fname, dpi=300):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "ch3-outline")

    ## get fname
    fname = save_dir / f"{fname}.pdf"

    # fig.savefig(fname, dpi=dpi)
    return

def get_sims_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sims = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## do simulation
        sim_y = model.simulate(fit_ds=params.sel(year=y), **simulation_kwargs)

        ## append
        sims.append(sim_y)

    ## put back in xarray
    sims = xr.concat(sims, dim=params.year)

    return sims

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)

## omit first year (bc of NaN in h,hw vars)
Th = Th.sel(time=slice("1851", None))

#### OHC

In [ ]:
def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom


forced_wide, anom_wide = load_consolidated_wide()

In [ ]:
lon_avg = lambda x, lons: x.sel(longitude=slice(*lons)).mean("longitude")

## compute ohc
Th["h_w_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 180)),
)["T"]

## compute ohc
Th["h_ohc"] = src.utils.reconstruct_wrapper(
    anom_wide[["T", "T_comp"]],
    lambda x: lon_avg(x.integrate("z_t"), (120, 280)),
)["T"]

### preprocess

In [ ]:
## should we remove median?
REMOVE_MEDIAN = True

## standardize (for convenience)
Th /= Th.std()

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

## remove median (optional)
if REMOVE_MEDIAN:
    median = Th_rolling.groupby("time.month").median(["time", "member"])
    Th_rolling = Th_rolling.groupby("time.month") - median

## Compute time-varying RO parameters

In [ ]:
## specify variables
varnames = ["T_3", "h_w_ohc"]

## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)
fit_kwargs = dict(
    # ac_mask_idx=None, maskb=["h"], maskNT=["T2", "T3", "TH"], maskNH=["T2"]
    # ac_mask_idx=None, maskNT=["T2", "T3"], maskNH=[]
    ac_mask_idx=None, maskNT=["T2", "T3"], maskNH=[]
)

## get fits
fits = get_fits_over_time_wrapper(
    Th_rolling[varnames],
    model=MODEL,
    by_member=False,
    # fname="T_3-h_w_ohc-zero_median_all-members.nc",
    fname=None,
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps = fits["Lac"].isel(ranky=1, rankx=1)
coefs = xr.concat([R, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

## get coefficient array
z = np.linspace(-3, 3)
Z = np.stack([z**i for i in range(3)], axis=1)
Z = xr.DataArray(Z, coords=dict(T=z, form=coefs.form), dims=["T", "form"])

## reconstruct nonlinear R
R_nl = (coefs * Z).sum("form")
R_nl_ann = R_nl.mean("cycle")
eps_ann = eps.mean("cycle")

In [ ]:
def add_vticks(axs, xticks, xlines=None):
    """add vertical lines to axs"""

    ## specify line style
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)

    ## loop thru axs
    for ax in axs:
        ax.set_xticks(xticks)
        if xlines is not None:
            for x0 in xlines:
                ax.axvline(x0, **ax_kwargs)

    return

In [ ]:
YEARS = [1871, 2011, 2051, 2081]
colors = cmocean.cm.balance(np.linspace(0.2, 0.8, len(YEARS)))

fig, axs = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
for c, y in zip(colors, YEARS):
    axs[0].plot(R_nl["T"], R_nl_ann.sel(year=y), c=c, label=y)

## compute R coefficients
year = R_nl_ann.year
R_pos = R_nl_ann.where(R_nl_ann["T"] > 0).mean("T")
R_neg = R_nl_ann.where(R_nl_ann["T"] < 0).mean("T")
R_all = R_nl_ann.mean("T")

## plot R and R-epsilon
for R_, c, label in zip(
    [R_pos, R_neg, R_all], ["r", "b", "k"], [r"$T>0$", r"$T<0$", "All"]
):

    ## get shared args
    plot_kwargs = dict(c=c, label=label)

    axs[1].plot(year, R_, **plot_kwargs)
    axs[2].plot(year, R_ + eps_ann, **plot_kwargs)
## label
labels = [r"$R$ snapshots", r"$R$", r"$R-\varepsilon$"]
for ax, title in zip(axs, labels):
    ax.set_title(title)
axs[0].legend()
axs[2].legend(loc=[1, 0.5])
axs[0].set_xlabel(r"Niño 3")
axs[0].set_ylabel(r"Growth rate (yr$^{-1}$)")
axs[0].set_yticks([-1, 0, 1])
axs[1].set_yticks([0, 0.6, 1.2])
axs[2].set_yticks([-1, -1.6, -2.2])
axs[1].set_ylim([-0.2, 1.6])
axs[2].set_ylim([-2.4, -0.6])

## formatting
ax_kwargs = dict(ls="--", c="k", lw=0.8)
axs[0].axvline(0, **ax_kwargs)
axs[0].axhline(0, **ax_kwargs)
add_vticks(axs[1:], xticks=YEARS[:-1], xlines=YEARS[:-1])
for ax in axs[1:]:
    axs[1].axhline(0, **ax_kwargs)

plt.show()

## Validate RO model

### Stochastic simulation

In [ ]:
# simulation specs
simulation_kwargs = dict(
    nyear=40,
    ncopy=500,
    seed=1000,
    X0_ds=Th_rolling[varnames].isel(year=0, member=0, time=0),
    noise_type="white",
    use_noise_cov=False,
    is_xi_stdac=False,
)

model = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)

In [ ]:
## do simulations
sims = get_sims_over_time(model=model, params=fits, **simulation_kwargs)

## resample to DJF
get_djf = lambda x : x.resample({"time":"QS-DEC"}).mean().isel(time=slice(4,-4,4))

## resample to seasonal
sims_djf = get_djf(sims)
Th_djf = get_djf(Th_rolling[varnames])

### Compute stats

In [ ]:
def get_std(x):
    return x.std(["time","member"])

def get_quantiles(x):
    return x.quantile(dim=["time","member"],q=[.05,.5,.95])

def get_stats(x):
    """compute relevant stats for given dataset"""

    ## empty dataset to hold results
    stats = xr.Dataset()

    ## helper func to convert to xr.DataArray
    to_da = lambda x : x.to_dataarray(dim="v")

    ## compute
    stats["sigma"] = to_da(get_std(x))
    stats["q"] = to_da(get_quantiles(x))

    return stats

In [ ]:
stats_ro = get_stats(sims_djf)
stats_gt = get_stats(Th_djf)

### Plot results

#### Quantiles

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(6, 2.5))
for ax, stats_ in zip(axs, [stats_gt, stats_ro]):
    q = stats_["q"].sel(v="T_3")
    
    ax.plot(stats_.year, q.sel(quantile=0.95), c="r")
    ax.plot(stats_.year, -q.sel(quantile=0.05), c="b")
    ax.plot(stats_.year, .5*(q.sel(quantile=.95)-q.sel(quantile=0.05)), c="k")

    ax_kwargs = dict(ls="--", c="gray", lw=.8)
    for y in [2010,2050]:
        ax.axvline(y, **ax_kwargs)

src.utils.set_lims(axs)
plt.show()

#### PDFs

In [ ]:
def get_pdf(x, year, varname="T_3", edges=np.linspace(-4.5,4.5,20)):
    """compute pdf for given data"""

    ## reshape data
    x_ = x[varname].sel(year=year).stack(s=["time", "member"]).values

    ## compute
    pdf, _ = src.utils.get_empirical_pdf(x_, edges=edges)

    return pdf, edges

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(6, 2.5), layout="constrained")

for ax, x, title in zip(axs, [Th_djf, sims_djf], ["CESM2", "RO"]):

    

    for y, kwargs in zip(
        [1871, 2011, 2081],
        [dict(color="gray", fill=True, alpha=.2), dict(lw=2), dict(lw=2)],
    ):

        ## compute pdf for given year
        pdf_y, edges = get_pdf(x, year=y)

        ## plot
        ax.stairs(pdf_y, edges, label=y, **kwargs)

    ## label/format
    ax.set_title(title)
    ax.axvline(0, ls="--", c="k", lw=0.8)
    ax.set_xlim([-4.3,4.3])
    ax.set_xticks([-3,0,3])

## formatting
src.utils.set_lims(axs)
axs[1].set_yticks([])
axs[1].legend(prop=dict(size=10))
# axs[0].set-
plt.show()

## Old

### estimate variance of CESM over time

In [ ]:
## compute rolling std
# Th_std = get_rolling_std(Th, n=20)
Th_std = Th_rolling.groupby("time.month").std("time")

## compute percentage change in std
baseline = Th_std.isel(year=0).mean("member")
delta_Th_std = 100 * (Th_std - baseline) / baseline

### Compare model and RO

Function to plot results

In [ ]:
def plot_stats_comp(ax, list_of_stats, labels, colors=None, n=varnames[0]):
    """plot comparison of variance over time"""

    if colors is None:
        colors = sns.color_palette()[: len(list_of_stats)]

    for stats, label, c in zip(list_of_stats, labels, colors):

        ## plot median
        mplot = ax.plot(stats.year, stats[n].sel(q=0.5), lw=2.5, label=label, c=c)

        ## plot lower/upper quantiles
        kwargs = dict(c=mplot[0].get_color(), lw=0.8)
        for q in stats.q:
            if q != 0.5:
                ax.plot(stats.year, stats[n].sel(q=q), **kwargs)

    ## label and set plotting specs
    ax.set_xlabel("Year")
    ax.set_ylabel(r"$\sigma_T$ ($^{\circ}$C)")
    ax.set_ylim([0.3, 1.7])
    ax.set_xticks([1870, 1975, 2080])
    ax.set_yticks([0.6, 1.2])

    return


def format_validation_plots(axs):
    """add formatting to CESM v. RO plot"""

    axs[1, 0].set_ylabel(r"$\sigma_h$($^{\circ}$C)")
    axs[0, 1].legend(prop=dict(size=8))
    for i in range(axs.shape[1]):
        ## remove ticks
        axs[0, i].set_xticks([])
        axs[0, i].set_xlabel(None)
        if i > 0:
            for ax in axs[:, i]:
                ax.set_yticks([])
                ax.set_ylabel(None)

    return

Make the plot

In [ ]:
## specify months to plot
PLOT_MONTHS = [2, 5, 8, 11]

fig, axs = plt.subplots(2, 4, figsize=(8, 4), layout="constrained")

for i, m in enumerate(PLOT_MONTHS):

    ## compute stats
    stats_mpi = get_stats(Th_std).sel(month=m)
    stats_ro_v2 = get_stats(RO_sigma_over_time_v2).sel(month=m)

    ## specify kwargs
    plot_kwargs = dict(
        list_of_stats=[stats_mpi, stats_ro_v2],
        labels=["CESM", "RO"],
        colors=["k", sns.color_palette()[1]],
    )

    ## plot comparison
    for j, ax in enumerate(axs[:, i]):
        plot_stats_comp(ax, n=varnames[j], **plot_kwargs)

    ## label
    axs[0, i].set_title(calendar.month_name[m])

## format all subplots
format_validation_plots(axs)

# for ax in axs.flatten():
#     ax.set_ylim([None,2])

## save to file
# save(fig, "sigma_validation-ro")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
delta = lambda x: x / x.isel(year=0) - 1
plt.plot(params.year, delta(params["xi_T"].mean("cycle")))
plt.show()

In [ ]:
V0 = "eli_05_scaled"
V1 = "T_3"

sel = lambda x: src.utils.sel_month(x, 1).stack(s=["member", "time"])
fig, axs = plt.subplots(1, 2, figsize=(6.5, 3), layout="constrained")
axs[0].scatter(
    sel(Th_rolling[V0].isel(year=0)),
    sel(Th_rolling[V1].isel(year=0)),
    s=3,
    alpha=0.5,
)

axs[1].scatter(
    sel(Th_rolling[V0].isel(year=-1)),
    sel(Th_rolling[V1].isel(year=-1)),
    s=3,
    alpha=0.5,
)

src.utils.set_lims(axs)

plt.show()

### Plot diagnostics

### Snapshots of parameters over time

In [ ]:
def format_params_line(axs):
    """format line plots of parameters"""
    for ax in axs:

        ax.axhline(0, ls="--", c="k", lw=0.8)
        ax.set_xticks([2, 11])

    return

In [ ]:
## specify colormap and norm
CMAP = cmocean.cm.amp
CMAP_NORM = plt.Normalize(vmin=-1, vmax=2)

## specify years to plot
# YEARS = [1871, 2001, 2041, 2081]
YEARS = [1871, 2011, 2081]
# YEARS = [1871, 2021, 2081]
# YEARS = [1871, 1911, 1951, 1991]
# YEARS = [2021, 2041, 2061, 2081]

## Plot variance, noise, bjerknes index, and period
fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

## plot data
for i, y in enumerate(YEARS):

    ## variance
    axs[0].plot(
        Th_std.month,
        Th_std[varnames[0]].sel(year=y, method="nearest").mean("member"),
        c=CMAP(CMAP_NORM(i)),
        label=y,
        lw=2,
    )

    ## noise/BJ
    for ax, p in zip(axs[1:], ["xi_T_norm", "BJ_ac", "wyrtki"]):
        ax.plot(params.cycle, params[p].sel(year=y), c=CMAP(CMAP_NORM(i)), label=y)
        ax.set_title(p)


## formatting/label
format_params_line(axs)
axs[0].set_title(r"$\sigma(T)$")
axs[0].legend(prop=dict(size=10), loc="lower right")
axs[2].legend(prop=dict(size=10))  # , loc="upper left")

for ax in axs:
    ax.axvline(6)

plt.show()

## same, but for individual RO parameters
fig, axs = plt.subplots(1, 4, figsize=(8, 2), layout="constrained")

## loop thru parameters and years
for ax, p in zip(axs, ["R", "epsilon", "F1", "F2"]):
    for i, y in enumerate(YEARS):

        ## plot data
        ax.plot(params.cycle, params[p].sel(year=y), c=CMAP(CMAP_NORM(i)))

        ## formatting
        ax.set_title(p)


## formatting
format_params_line(axs)
for ax in axs:
    ax.axvline(5)
plt.show()

### Hovmoller plots for variance, growth rate, and noise

In [ ]:
def format_params_hov(axs):
    """format hovmoller axes"""

    for ax in axs:
        ax.set_xticks([1, 7, 12], labels=["Jan", "Jul", "Dec"])
        ax.axvline(7, c="w", ls="--", lw=1, alpha=0.8)
        ax.set_xlabel("Month")
        ax.axhline(2025, c="w", ls="--", lw=1)

    axs[0].set_ylabel("Year")
    axs[0].set_yticks([1870, 1975, 2080])

    for ax in axs[1:]:
        ax.set_yticks([])
        ax.set_ylim(axs[0].get_ylim())

    return

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(8, 3.5), layout="constrained")

#### plot change in std dev
plot_kwargs = dict(
    cmap="cmo.balance", levels=src.utils.make_cb_range(30, 6), extend="both"
)

## plot data
cp0 = axs[0].contourf(
    delta_Th_std.month,
    delta_Th_std.year,
    delta_Th_std["T_3"].mean("member").transpose("year", ...),
    **plot_kwargs
)

##### plot change in model params

## specify plotting specs
plot_kwargs = dict(
    cmap="cmo.balance", levels=src.utils.make_cb_range(1, 0.1), extend="both"
)

## plot data
cp1 = axs[1].contourf(
    params.cycle, params.year, delta_params["xi_T_norm"], **plot_kwargs
)
cp2 = axs[2].contourf(params.cycle, params.year, delta_params["BJ_ac"], **plot_kwargs)
cp3 = axs[3].contourf(params.cycle, params.year, delta_params["wyrtki"], **plot_kwargs)

## add colorbar
cb0 = fig.colorbar(cp0, label=r"% change", ticks=[-30, 0, 30])
cb1 = fig.colorbar(cp3, label=r"yr$^{-1}$", ticks=[-2, 0, 2])

## label])
axs[0].set_title(r"$\frac{\Delta \sigma(T)}{\sigma(T)_{1870}}$", size=10)
axs[1].set_title(r"$\Delta\left(\frac{\text{Noise}}{\sigma(T)}\right)$", size=10)
axs[2].set_title(r"$\Delta$ BJ", size=10)
axs[3].set_title(r"$\Delta$ Wyrtki", size=10)

## formatting
format_params_hov(axs)

plt.show()

### Same, but for RO parameters

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(7, 3.5), layout="constrained")


##### plot change in model params

## specify plotting specs
plot_kwargs = dict(
    cmap="cmo.balance", levels=src.utils.make_cb_range(2, 0.2), extend="both"
)

## plot data
cp0 = axs[0].contourf(params.cycle, params.year, delta_params["R"], **plot_kwargs)
cp1 = axs[1].contourf(params.cycle, params.year, delta_params["epsilon"], **plot_kwargs)
cp2 = axs[2].contourf(params.cycle, params.year, delta_params["F1"], **plot_kwargs)
cp3 = axs[3].contourf(params.cycle, params.year, delta_params["F2"], **plot_kwargs)

## add colorbar
cb1 = fig.colorbar(cp3, ticks=[-2, 0, 2])

## label
axs[0].set_ylabel("Year")
axs[0].set_yticks([1870, 1975, 2080])
axs[0].set_title(r"$\Delta R$", size=10)
axs[1].set_title(r"$\Delta \epsilon$", size=10)
axs[2].set_title(r"$\Delta F_1$", size=10)
axs[3].set_title(r"$\Delta F_2$", size=10)

## formatting
format_params_hov(axs)

plt.show()

### Change in annual-mean parameters over time

Plotting funcs

In [ ]:
def plot_curve(ax, x, **plot_kwargs):
    """plot change in parameter over time on given ax"""
    plot_data = ax.plot(x.year, x, lw=2, **plot_kwargs)

    return plot_data


def format_pot_ax(ax):
    """add formatting to parameter-over-time plot"""
    ax.set_xlim([None, None])
    ax.legend(prop=dict(size=10), loc="upper left")
    ax.set_xticks([1870, 1975, 2080])
    ax.set_yticks([-0.5, 0, 0.5])
    ax.set_ylabel(r"year$^{-1}$")
    ax.set_xlabel("Year")
    # ax.set_title(r"$\Delta \left(\text{RO parameters}\right)$")
    ax.axhline(0, c="k", ls="-", lw=0.5)
    return


def plot_pot0(ax, dp):
    """Plot change in parameters over time on ax object. 'dp' is change in params"""

    ## Plot Bjerknes growth rate
    plot_curve(ax, dp["BJ_ac"].mean("cycle"), c="k", label=r"$R-\varepsilon$")
    plot_curve(
        ax, dp["wyrtki"].mean("cycle"), c="k", ls="--", label=r"$\sqrt{F_1 F_2}$"
    )

    ## plot noise
    plot_curve(
        ax,
        dp["xi_T_norm"].mean("cycle"),
        label=r"$\xi_T/\sigma_T$",
        c="darkgray",
    )
    plot_curve(
        ax, dp["xi_h_norm"].mean("cycle"), label=r"$\xi_h/\sigma_h$", c="lightgray"
    )

    ## set axis specs
    format_pot_ax(ax)

    return


def plot_pot1(ax, dp):
    """Plot change in parameters over time on ax object. 'dp' is change in params"""

    ## plot params
    plot_curve(ax, dp["F1"].mean("cycle"), label=r"$F_1$", ls="--")
    plot_curve(ax, dp["R"].mean("cycle"), label=r"$R$")
    plot_curve(ax, dp["F2"].mean("cycle"), label=r"$F_2$", ls="--")
    plot_curve(ax, dp["epsilon"].mean("cycle"), label=r"$\varepsilon$")

    ## set axis specs
    format_pot_ax(ax)

    return

Make plot

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6.5, 3))

## plot change over time of parameters
plot_pot0(axs[0], dp=delta_params)
plot_pot1(axs[1], dp=delta_params)
axs[1].set_yticks([])
axs[1].set_ylabel(None)
src.utils.set_lims(axs)

for ax in axs:
    ax.axvline(2020, c="k", lw=0.8)

axs[0].set_ylim([-0.5, 0.8])

## save to file
# save(fig, "delta-params_ro")

plt.show()